# 🪄 Chạy OpenAvatarChat miễn phí trên Colab GPU

Notebook này tự động:
1. Cài `uv` + Python 3.11 (đúng yêu cầu của dự án) và clone `OpenAvatarChat`.
2. Cài dependencies (torch, fastapi, các model handler...).
3. Vá file config: đổi `RtcClient` (WebRTC) → `WsClient` (WebSocket) để khớp với client React đã làm ở panel **My AI Avatar**, tắt SSL tự ký, tự điền LLM/giọng đọc bạn chọn.
4. Tải model ASR/TTS/LLM handler cần thiết.
5. Chạy server ở cổng `8282` và mở **Cloudflare Quick Tunnel** miễn phí để lấy 1 URL `wss://...` public — dán URL đó vào ô "Server URL" trong panel.

> ⚠️ **Lưu ý quan trọng**: đây là môi trường Colab miễn phí — phiên sẽ tự ngắt sau vài giờ hoặc khi idle, và URL tunnel sẽ đổi mỗi lần chạy lại. Chỉ phù hợp để **test/demo**, không dùng cho sản phẩm chạy 24/7. GPU T4 miễn phí có thể chạy chậm hơn GPU cao cấp.

**Trước khi chạy**: vào `Runtime → Change runtime type → GPU (T4)` để bật GPU miễn phí.

In [ ]:
#@title Bước 0 — Kiểm tra GPU
!nvidia-smi || echo '⚠️ Chưa bật GPU. Vào Runtime > Change runtime type > GPU rồi chạy lại.'

In [ ]:
#@title Bước 1 — Cấu hình { display-mode: "form" }
GITHUB_REPO_URL = "https://github.com/HumanAIGC-Engineering/OpenAvatarChat.git" #@param {type:"string"}

#@markdown **LLM (bắt buộc)** — dùng API tương thích OpenAI. Có thể dùng OpenAI, DashScope (Qwen), hoặc Gemini OpenAI-compatible endpoint.
LLM_API_URL = "https://api.openai.com/v1" #@param {type:"string"}
LLM_API_KEY = "" #@param {type:"string"}
LLM_MODEL_NAME = "gpt-4o-mini" #@param {type:"string"}
SYSTEM_PROMPT = "B\u1ea1n l\u00e0 m\u1ed9t tr\u1ee3 l\u00fd AI, tr\u1ea3 l\u1eddi ng\u1eafn g\u1ecdn 2-3 c\u00e2u b\u1eb1ng ti\u1ebfng Vi\u1ec7t." #@param {type:"string"}

#@markdown **Giọng đọc TTS (Edge TTS, miễn phí, không cần tải model nặng)** — ví dụ giọng nữ Việt: `vi-VN-HoaiMyNeural`, giọng nam: `vi-VN-NamMinhNeural`.
TTS_VOICE = "vi-VN-HoaiMyNeural" #@param {type:"string"}

#@markdown Tắt render video avatar (LiteAvatar) để tiết kiệm VRAM GPU free — client React hiện chỉ dùng audio + text nên vẫn hoạt động đầy đủ. Nếu server lỗi khi khởi động, thử bỏ tick rồi chạy lại từ Bước 4.
DISABLE_AVATAR_VIDEO = True #@param {type:"boolean"}

print('Đã lưu cấu hình.')

In [ ]:
#@title Bước 2 — Clone repo + cài `uv`
%cd /content
!rm -rf OpenAvatarChat
!git clone --depth 1 {GITHUB_REPO_URL} OpenAvatarChat
%cd /content/OpenAvatarChat

# silero_vad là git submodule — clone --depth 1 không tự tải, phải init thủ công
# (nếu sau này bật thêm handler khác cũng là submodule, ví dụ LiteAvatar, cần init thêm dòng tương ứng)
!git submodule update --init src/handlers/vad/silerovad/silero_vad

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version

In [ ]:
#@title Bước 3 — Tạo venv Python 3.11 và cài dependencies (mất vài phút vì tải torch/CUDA)
!uv venv --python 3.11.7 .venv
!uv pip install --python .venv/bin/python --editable .

# QUAN TRỌNG: 'kích hoạt' venv cho cả session Colab (kể cả các lệnh gọi uv
# ngầm bên trong install.py ở Bước 5) — nếu không, uv sẽ tự rơi về Python
# hệ thống của Colab thay vì .venv/bin/python, gây lệch môi trường khi chạy demo.py.
import os
venv_path = os.path.abspath(".venv")
os.environ["VIRTUAL_ENV"] = venv_path
os.environ["PATH"] = os.path.join(venv_path, "bin") + ":" + os.environ["PATH"]
print("VIRTUAL_ENV =", os.environ["VIRTUAL_ENV"])

In [ ]:
#@title Bước 4 — Vá config: RtcClient → WsClient, tắt SSL tự ký, điền LLM/giọng đọc
import pathlib

src_path = pathlib.Path("config/chat_with_openai_compatible_edge_tts.yaml")
text = src_path.read_text(encoding="utf-8")

rtc_block = (
    "      RtcClient:\n"
    "        module: client/rtc_client/client_handler_rtc\n"
    "        # max time a session will last for\n"
    "        connection_ttl: 900\n"
)
ws_block = (
    "      WsClient:\n"
    "        module: client/ws_client/ws_client_handler\n"
    "        connection_ttl: 900\n"
)

if rtc_block not in text:
    raise SystemExit(
        "Không tìm thấy khối 'RtcClient' đúng như dự kiến — có thể repo gốc đã đổi cấu trúc.\n"
        "Mở config/chat_with_openai_compatible_edge_tts.yaml và tự đổi thủ công:\n"
        "  RtcClient: -> module: client/rtc_client/client_handler_rtc\n"
        "thành:\n"
        "  WsClient: -> module: client/ws_client/ws_client_handler"
    )
text = text.replace(rtc_block, ws_block)

# Tắt SSL tự ký để cloudflared xử lý HTTPS/WSS ở tầng ngoài (server chạy plain http/ws nội bộ)
text = text.replace('cert_file: "ssl_certs/localhost.crt"', 'cert_file: ""')
text = text.replace('cert_key: "ssl_certs/localhost.key"', 'cert_key: ""')

if DISABLE_AVATAR_VIDEO:
    text = text.replace(
        "      LiteAvatar:\n        module: avatar/liteavatar/avatar_handler_liteavatar",
        "      LiteAvatar:\n        enabled: False\n        module: avatar/liteavatar/avatar_handler_liteavatar",
    )

text = text.replace('model_name: "qwen-plus"', f'model_name: "{LLM_MODEL_NAME}"')
text = text.replace(
    'api_url: "https://dashscope.aliyuncs.com/compatible-mode/v1"',
    f'api_url: "{LLM_API_URL}"',
)
text = text.replace(
    'system_prompt: "\u8bf7\u4f60\u626e\u6f14\u4e00\u4e2a AI \u52a9\u624b\uff0c\u7528\u7b80\u77ed\u7684\u4e24\u4e09\u53e5\u5bf9\u8bdd\u6765\u56de\u7b54\u7528\u6237\u7684\u95ee\u9898\uff0c\u5e76\u5728\u5bf9\u8bdd\u5185\u5bb9\u4e2d\u52a0\u5165\u5408\u9002\u7684\u6807\u70b9\u7b26\u53f7\uff0c\u4e0d\u9700\u8981\u8ba8\u8bba\u6807\u70b9\u7b26\u53f7\u76f8\u5173\u7684\u5185\u5bb9"',
    f'system_prompt: "{SYSTEM_PROMPT}"',
)
text = text.replace('voice: "zh-CN-XiaoxiaoNeural"', f'voice: "{TTS_VOICE}"')

if LLM_API_KEY:
    os.environ["DASHSCOPE_API_KEY"] = LLM_API_KEY  # default env var the handler reads
    text = text.replace(
        "        api_url:",
        f'        api_key: "{LLM_API_KEY}"\n        api_url:',
    )

pathlib.Path("config/colab_config.yaml").write_text(text, encoding="utf-8")
print("Đã tạo config/colab_config.yaml:\n")
print(text)

In [ ]:
#@title Bước 4b — Vá install.py để ép dùng đúng .venv (khắc phục uv tự rơi về Python hệ thống)
import pathlib

install_path = pathlib.Path("install.py")
install_text = install_path.read_text(encoding="utf-8")

target = '["uv", "pip", "install"'
patched = '["uv", "pip", "install", "--python", ".venv/bin/python"'

count = install_text.count(target)
if count == 0:
    print("⚠️ Không tìm thấy chuỗi cần vá — có thể install.py đã đổi cấu trúc, kiểm tra thủ công.")
else:
    install_text = install_text.replace(target, patched)
    install_path.write_text(install_text, encoding="utf-8")
    print(f"Đã vá {count} lệnh 'uv pip install' bên trong install.py để luôn chỉ định --python .venv/bin/python")

In [ ]:
#@title Bước 5 — Tải model (ASR SenseVoice...). Có thể mất vài phút tùy đường truyền.
!.venv/bin/python install.py --config config/colab_config.yaml

In [ ]:
#@title Bước 6 — Cài Cloudflare Tunnel (miễn phí, không cần tài khoản)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
#@title Bước 7 — Chạy server + mở tunnel, lấy URL wss:// để dán vào panel React
import re
import socket
import subprocess
import time

server_log = open("/content/server.log", "w")
server_proc = subprocess.Popen(
    [".venv/bin/python", "src/demo.py", "--config", "config/colab_config.yaml"],
    stdout=server_log, stderr=subprocess.STDOUT, cwd="/content/OpenAvatarChat",
)
print(f"Server PID: {server_proc.pid} — đang khởi động, chờ tới khi cổng 8282 sẵn sàng (tối đa 5 phút)...")

def port_is_open(host, port, timeout=1.0):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

MAX_WAIT_SECONDS = 300
waited = 0
server_ready = False
while waited < MAX_WAIT_SECONDS:
    if server_proc.poll() is not None:
        print(f"❌ Server đã thoát sớm (exit code {server_proc.returncode}) — xem server.log bên dưới:")
        print(open("/content/server.log").read()[-4000:])
        break
    if port_is_open("localhost", 8282):
        server_ready = True
        print(f"✅ Server đã mở cổng 8282 sau ~{waited}s.")
        break
    time.sleep(3)
    waited += 3
    if waited % 15 == 0:
        print(f"  ... vẫn đang chờ ({waited}s) — model ASR/TTS/LLM có thể cần thời gian tải/nạp GPU lần đầu.")

if not server_ready:
    if server_proc.poll() is None:
        print("⚠️ Hết thời gian chờ nhưng server vẫn đang chạy — có thể cần thêm thời gian. Xem server.log:")
        print(open("/content/server.log").read()[-4000:])
    raise SystemExit("Dừng lại: server chưa sẵn sàng, đừng mở tunnel để tránh nhầm lẫn URL chết.")

tunnel_log = open("/content/tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8282"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    log_text = open("/content/tunnel.log").read()
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break

if public_url:
    ws_url = public_url.replace("https://", "wss://")
    print("✅ Server + tunnel đã sẵn sàng!\n")
    print(f"Dán URL này vào ô 'Server URL' trong panel My AI Avatar:\n\n   {ws_url}\n")
else:
    print("⚠️ Chưa lấy được URL tunnel. Kiểm tra log bên dưới:")
    print(open("/content/tunnel.log").read())
    print("\n--- server.log ---")
    print(open("/content/server.log").read()[-3000:])

In [ ]:
#@title (Tuỳ chọn) Xem log server / tunnel trực tiếp để debug
print("----- server.log (300 dòng cuối) -----")
!tail -n 300 /content/server.log
print("\n----- tunnel.log -----")
!cat /content/tunnel.log

In [ ]:
#@title (Tuỳ chọn) Dừng server + tunnel
try:
    server_proc.terminate()
    tunnel_proc.terminate()
    print("Đã dừng server và tunnel.")
except NameError:
    print("Chưa có process nào đang chạy trong session này.")

## Ghi chú
- Mỗi lần Colab ngắt phiên (đóng tab, hết giờ, idle quá lâu), bạn phải chạy lại **toàn bộ notebook từ Bước 2** và URL tunnel ở Bước 7 sẽ **thay đổi** — nhớ cập nhật lại ô "Server URL" trong panel React.
- Nếu dùng `LLM_API_URL` của DashScope (Qwen), cần API key từ Alibaba Cloud (biến `DASHSCOPE_API_KEY`). Nếu dùng OpenAI, điền `LLM_API_KEY` = OpenAI key, `LLM_API_URL` = `https://api.openai.com/v1`, `LLM_MODEL_NAME` = ví dụ `gpt-4o-mini`.
- Đây là cấu hình rút gọn (Edge TTS thay vì CosyVoice, tắt LiteAvatar) để vừa với GPU T4 free-tier. Muốn avatar video/motion data thật, cần renderer WebGL riêng của dự án (chưa có trong client React hiện tại) và GPU mạnh hơn.